# 02 Optimizacion

Este notebook construye el problema completo de optimizacion y ejecuta los dos metodos principales del proyecto:
- ACO
- Algoritmo genetico

La logica reusable vive en `src/`. El notebook solo orquesta, documenta y muestra resultados.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

WindowsPath('C:/Carlos/Uni/Algortimos/trabajos/trabajo_1/trabajo_carlos/2. optimizacion_combinatoria')

In [2]:
from dataclasses import asdict

import numpy as np
import pandas as pd

from src.config_modelo_costo import model_metadata
from src.core_grafo import load_city_index_to_name, save_json
from src.core_matrices import build_complete_matrices
from src.core_paths import ACO_OUTPUT_DIR, GA_OUTPUT_DIR, INPUT_COST_MATRIX_PATH, WEIGHTED_EDGES_CSV_PATH
from src.core_tsp_aco import ACOConfig, run_aco
from src.core_tsp_ga import GAConfig, run_ga


## 1. Construccion de matrices completas

A partir del grafo ponderado de conexiones directas se construyen las matrices de costos minimos y la matriz de siguiente salto para reconstruccion de rutas.

In [3]:
matrices = build_complete_matrices()
matrices['summary']

{'ciudades': 96,
 'shape': [96, 96],
 'costo_minimo_finito': 11.968667,
 'costo_maximo_finito': 759.442332,
 'costo_promedio_finito': 232.18437044078948,
 'diagonal_cero': True,
 'formula_peso': 'peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02',
 'vehiculo_referencia': {'name': 'Renault Clio',
  'fuel_type': 'gasolina',
  'segment': 'compacto',
  'note': 'Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.'},
 'tarifa_vendedor_eur_h': 12.02,
 'salario_referencia': 'SMIC Francia 2026'}

In [4]:
cost_matrix = matrices['cost_matrix']
print('Shape matriz completa:', cost_matrix.shape)
print('Costo minimo finito:', np.min(cost_matrix[np.isfinite(cost_matrix)]))
print('Costo maximo finito:', np.max(cost_matrix[np.isfinite(cost_matrix)]))

Shape matriz completa: (96, 96)
Costo minimo finito: 0.0
Costo maximo finito: 759.442332


In [5]:
city_name_map = load_city_index_to_name(WEIGHTED_EDGES_CSV_PATH)
city_names = [city_name_map[index] for index in range(cost_matrix.shape[0])]
df_cost_preview = pd.DataFrame(cost_matrix, index=city_names, columns=city_names)
df_cost_preview.iloc[:5, :5]

,Bourg-en-Bresse,Laon,Moulins,Digne-les-Bains,Gap
Bourg-en-Bresse,0.000000,222.440001,89.836000,186.296001,147.731334
Laon,222.440001,0.000000,188.506000,402.643334,364.078667
Moulins,89.836000,188.506000,0.000000,237.889000,199.324333
Digne-les-Bains,186.296001,402.643334,237.889000,0.000000,38.564667
Gap,147.731334,364.078667,199.324333,38.564667,0.000000


## 2. Validacion de la matriz TSP

La matriz consolidada ya sale lista para TSP: es cuadrada, tiene diagonal en cero y mantiene costos finitos fuera de la diagonal.

In [6]:
print('Shape matriz TSP:', cost_matrix.shape)
print('Diagonal cero:', np.allclose(np.diag(cost_matrix), 0.0))
print('Finita fuera diagonal:', np.isfinite(cost_matrix[~np.eye(cost_matrix.shape[0], dtype=bool)]).all())

Shape matriz TSP: (96, 96)
Diagonal cero: True
Finita fuera diagonal: True


In [7]:
aco_matrix = np.load(INPUT_COST_MATRIX_PATH)
print('Shape matriz TSP cargada desde disco:', aco_matrix.shape)
print('Coincide con la matriz construida:', np.allclose(aco_matrix, cost_matrix))

Shape matriz TSP cargada desde disco: (96, 96)
Coincide con la matriz construida: True


## 3. Ejecucion de ACO

Se corre la colonia de hormigas sobre la matriz completa preparada.

In [8]:
aco_config = ACOConfig()
best_tour_aco, best_cost_aco, history_aco = run_aco(aco_matrix, aco_config)

aco_result = {
    'configuracion_aco': asdict(aco_config),
    'shape_matriz': list(aco_matrix.shape),
    'mejor_costo_tour': round(best_cost_aco, 6),
    'mejor_tour_indices': best_tour_aco,
    'mejor_tour_indices_sin_cierre': best_tour_aco[:-1],
    'mejor_tour_nombres': [city_names[index] for index in best_tour_aco],
    'mejor_tour_nombres_sin_cierre': [city_names[index] for index in best_tour_aco[:-1]],
    'historial_mejor_costo': [round(value, 6) for value in history_aco],
}
aco_result.update(model_metadata())

aco_summary = {
    'mejor_costo_tour': round(best_cost_aco, 6),
    'iteraciones': aco_config.iterations,
    'hormigas': aco_config.ants,
    'seed': aco_config.seed,
    'n_ciudades': len(best_tour_aco) - 1,
    'tour_cerrado': len(best_tour_aco),
}
aco_summary.update(model_metadata())

save_json(ACO_OUTPUT_DIR / 'resultado_aco_final.json', aco_result)
save_json(ACO_OUTPUT_DIR / 'resumen_aco_final.json', aco_summary)

aco_summary

{'mejor_costo_tour': 4471.569332,
 'iteraciones': 250,
 'hormigas': 40,
 'seed': 42,
 'n_ciudades': 96,
 'tour_cerrado': 97,
 'vehiculo_referencia': {'name': 'Renault Clio',
  'fuel_type': 'gasolina',
  'segment': 'compacto',
  'note': 'Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.'},
 'tarifa_vendedor_eur_h': 12.02,
 'salario_referencia': 'SMIC Francia 2026',
 'formula_peso': 'peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02'}

In [9]:
pd.DataFrame({'iteracion': range(1, len(history_aco) + 1), 'mejor_costo': history_aco}).head()

,iteracion,mejor_costo
0,1,6855.457334
1,2,6670.609333
2,3,6549.810999
3,4,6549.810999
4,5,6549.810999


## 4. Ejecucion del algoritmo genetico

Se corre GA sobre la misma matriz. Si existe un resultado ACO, se usa como semilla inicial cuando la configuracion lo permite.

In [10]:
ga_config = GAConfig()
best_tour_ga, best_cost_ga, history_ga = run_ga(
    aco_matrix,
    ga_config,
    ACO_OUTPUT_DIR / 'resultado_aco_final.json',
)

ga_result = {
    'configuracion_ga': asdict(ga_config),
    'shape_matriz': list(aco_matrix.shape),
    'mejor_costo_tour': round(best_cost_ga, 6),
    'mejor_tour_indices': best_tour_ga,
    'mejor_tour_indices_sin_cierre': best_tour_ga[:-1],
    'mejor_tour_nombres': [city_names[index] for index in best_tour_ga],
    'mejor_tour_nombres_sin_cierre': [city_names[index] for index in best_tour_ga[:-1]],
    'historial_mejor_costo': [round(value, 6) for value in history_ga],
}
ga_result.update(model_metadata())

ga_summary = {
    'mejor_costo_tour': round(best_cost_ga, 6),
    'generaciones': ga_config.generations,
    'tam_poblacion': ga_config.population_size,
    'seed': ga_config.seed,
    'n_ciudades': len(best_tour_ga) - 1,
    'tour_cerrado': len(best_tour_ga),
}
ga_summary.update(model_metadata())

save_json(GA_OUTPUT_DIR / 'resultado_ga_final.json', ga_result)
save_json(GA_OUTPUT_DIR / 'resumen_ga_final.json', ga_summary)

ga_summary

{'mejor_costo_tour': 4303.602999,
 'generaciones': 350,
 'tam_poblacion': 180,
 'seed': 42,
 'n_ciudades': 96,
 'tour_cerrado': 97,
 'vehiculo_referencia': {'name': 'Renault Clio',
  'fuel_type': 'gasolina',
  'segment': 'compacto',
  'note': 'Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.'},
 'tarifa_vendedor_eur_h': 12.02,
 'salario_referencia': 'SMIC Francia 2026',
 'formula_peso': 'peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02'}

In [11]:
pd.DataFrame({'generacion': range(1, len(history_ga) + 1), 'mejor_costo': history_ga}).head()

,generacion,mejor_costo
0,1,4471.569332
1,2,4471.569332
2,3,4471.569332
3,4,4471.569332
4,5,4440.005331


## 5. Comparacion rapida

Se deja una comparacion preliminar del costo final de ambos metodos. La visualizacion y seleccion final se desarrollaran en `03_resultados_finales.ipynb`.

In [12]:
comparison = pd.DataFrame([
    {'metodo': 'ACO', 'mejor_costo': best_cost_aco},
    {'metodo': 'GA', 'mejor_costo': best_cost_ga},
])
comparison['diferencia_vs_mejor'] = comparison['mejor_costo'] - comparison['mejor_costo'].min()
comparison.sort_values('mejor_costo').reset_index(drop=True)

,metodo,mejor_costo,diferencia_vs_mejor
0,GA,4303.602999,0.000000
1,ACO,4471.569332,167.966333


## Salida de esta etapa

Con esto queda listo el problema optimizado y quedan guardados los resultados base en:
- `data/processed`
- `outputs/aco`
- `outputs/ga`

El siguiente notebook se enfocara en expandir la mejor ruta, comparar formalmente ambos metodos y generar artefactos finales para la entrega y la web.